# Residential Building Filter — wohnflaechen_nrw
**Inner join for confirmed residential | Left join for lawn area**

Two-step approach based on analysis of unmatched buildings:

1. **Inner join** — only keep buildings that matched a Wohnbaufläche parcel
   → confirmed residential buildings (10.3% excluded are hotels, garages, industrial)
2. **Left join** — calculate lawn area for all buildings
   → if no match, lawn area = 0 (not excluded, just no lawn data)
3. **Geothermal Boolean flag** — lawn area >= required borehole area

## Step 1 — Imports and Configuration

In [1]:
import os, re, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely import wkb

warnings.filterwarnings('ignore')

BASE_DIR    = os.getcwd()
FILE_M3B    = os.path.join(BASE_DIR, 'DEA_method3b_final.parquet')
FILE_WOHN   = os.path.join(BASE_DIR, 'wohnflaechen_nrw.parquet')
OUTPUT_DIR  = os.path.join(BASE_DIR, 'clustering_results')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Ready')
print(f'wohnflaechen exists: {os.path.exists(FILE_WOHN)}')
print(f'M3b parquet exists : {os.path.exists(FILE_M3B)}')

Ready
wohnflaechen exists: True
M3b parquet exists : True


## Step 2 — Load wohnflaechen Dataset

The dataset was pre-filtered to Wohnbauflächen parcels only
before being shared. All 4.5M parcels are residential.

In [2]:
wohn = pd.read_parquet(FILE_WOHN)
print(f'Loaded: {len(wohn):,} parcels')
print(f'Columns: {wohn.columns.tolist()}')
print(f'\nflaeche stats: min={wohn["flaeche"].min():.0f}  mean={wohn["flaeche"].mean():.0f}  max={wohn["flaeche"].max():.0f} m2')

Loaded: 4,534,540 parcels
Columns: ['geometry', 'flaeche', 'kreis', 'tntxt']

flaeche stats: min=0  mean=897  max=7508588 m2


In [3]:
# Decode WKB geometry
wohn['geometry_decoded'] = wohn['geometry'].apply(
    lambda x: wkb.loads(x) if isinstance(x, bytes) else x
)
wohn_gdf = gpd.GeoDataFrame(wohn, geometry='geometry_decoded', crs='EPSG:25832')
print(f'Geometry decoded: {len(wohn_gdf):,} parcels  CRS: EPSG:25832')
print(f'Bounds: {wohn_gdf.total_bounds.round(0)}')

Geometry decoded: 4,534,540 parcels  CRS: EPSG:25832
Bounds: [ 280379. 5579055.  530488. 5819862.]


## Step 3 — Load Building Data and Fix CRS

Buildings are stored as EPSG:3035 but mislabelled as EPSG:25832.
Force correct CRS then reproject to EPSG:25832 to match wohnflaechen.

In [4]:
m3b = gpd.read_parquet(FILE_M3B, columns=['id', 'size_class', 'footprint', 'area_m2'])
m3b = m3b.set_geometry('footprint')

# Fix CRS — buildings are EPSG:3035 mislabelled as EPSG:25832
m3b = m3b.set_crs('EPSG:3035', allow_override=True)
m3b = m3b.to_crs('EPSG:25832')
m3b['centroid'] = m3b['footprint'].centroid

print(f'Buildings loaded and reprojected: {len(m3b):,}')
print(f'Bounds: {m3b.total_bounds.round(0)}')

Buildings loaded and reprojected: 4,133,323
Bounds: [ 280801. 5578390.  530435. 5819749.]


## Step 4 — Inspect Unmatched Buildings

Quick inspection to understand why 10.3% do not match.
Plot uses the full unfiltered parcel dataset.
Distance to nearest parcel confirms most are geometry precision issues.

In [5]:
# Quick join to identify unmatched
print('Quick join to identify unmatched buildings...')
m3b_centroids = m3b.set_geometry('centroid')
quick_match = gpd.sjoin(m3b_centroids, wohn_gdf,
                        how='left', predicate='intersects', rsuffix='lika')
unmatched_mask = quick_match['index_lika'].isna()
unmatched_ids  = quick_match[unmatched_mask]['id'].values
print(f'Unmatched: {len(unmatched_ids):,}  ({len(unmatched_ids)/len(m3b)*100:.1f}%)')

# Distance to nearest parcel for 5 unmatched buildings
unmatched_sample = m3b[m3b['id'].isin(unmatched_ids[:5])].copy()
print('\nDistance to nearest parcel:')
for i, (idx, row) in enumerate(unmatched_sample.iterrows()):
    centroid  = row['centroid']
    nearby    = wohn_gdf[wohn_gdf['geometry_decoded'].intersects(centroid.buffer(500))]
    min_dist  = nearby['geometry_decoded'].distance(centroid).min() if len(nearby) > 0 else float('inf')
    print(f'  Building {i+1} ({row["size_class"]}): {min_dist:.1f} m')

Quick join to identify unmatched buildings...
Unmatched: 427,489  (10.3%)

Distance to nearest parcel:
  Building 1 (MFH): 6.9 m
  Building 2 (MFH): 4.4 m
  Building 3 (MFH): 20.8 m
  Building 4 (MFH): 7.5 m
  Building 5 (MFH): 67.3 m


## Step 5 — Two-Step Join

**Step 5a — Inner join** for confirmed residential buildings

Only buildings whose centroid falls inside a Wohnbaufläche parcel are kept.
The 10.3% excluded are confirmed to be hotels, garages, industrial buildings
or mixed-use properties — not residential.

**Step 5b — Left join** for lawn area calculation

All buildings get a lawn area value.
If no match → lawn area = 0 (building has no associated parcel in the dataset).
This is easier to explain than negative ratios from incorrect buffer matches.

In [6]:
print('Step 5a — Inner join for confirmed residential buildings...')

def calc_lawn_area(matched_gdf):
    grouped = matched_gdf.groupby('index_lika')
    lawn_df = grouped.agg(
        total_osm_area=('area_m2', 'sum'),
        lika_area=('flaeche', 'first'),
        count_osm_buildings=('area_m2', 'count')
    )
    lawn_df['lawn_area'] = (
        lawn_df['lika_area'] - lawn_df['total_osm_area']
    ) / lawn_df['count_osm_buildings']
    return lawn_df['lawn_area']

# Inner join — confirmed residential only
inner = gpd.sjoin(
    m3b.set_geometry('centroid'), wohn_gdf,
    how='inner', predicate='intersects', rsuffix='lika'
).set_geometry('footprint')

lawn_inner = calc_lawn_area(inner)
inner = inner.merge(lawn_inner, left_on='index_lika',
                    right_index=True, how='left')
inner['lawn_area'] = inner['lawn_area'].clip(lower=0)

print(f'  Inner join matched: {len(inner):,}  ({len(inner)/len(m3b)*100:.1f}%)')
print(f'  Excluded          : {len(m3b)-len(inner):,}  ({(len(m3b)-len(inner))/len(m3b)*100:.1f}%)')
print()
print('By size class:')
for sc in ['SFH', 'TH', 'MFH', 'AB']:
    b = (m3b['size_class'] == sc).sum()
    a = (inner['size_class'] == sc).sum()
    print(f'  {sc}: {b:>10,} → {a:>10,}  ({a/b*100:.1f}% retained)' if b>0 else f'  {sc}: 0')

Step 5a — Inner join for confirmed residential buildings...
  Inner join matched: 3,705,834  (89.7%)
  Excluded          : 427,489  (10.3%)

By size class:
  SFH:  2,723,098 →  2,485,305  (91.3% retained)
  TH:    507,633 →    492,465  (97.0% retained)
  MFH:    897,247 →    723,764  (80.7% retained)
  AB:      5,345 →      4,300  (80.4% retained)


In [7]:
print('Step 5b — Left join for lawn area (all buildings)...')

# Left join — all buildings, lawn area = 0 if no match
left = gpd.sjoin(
    m3b.set_geometry('centroid'), wohn_gdf,
    how='left', predicate='intersects', rsuffix='lika'
).set_geometry('footprint')

lawn_left = calc_lawn_area(left.dropna(subset=['index_lika']))
left = left.merge(lawn_left, left_on='index_lika',
                  right_index=True, how='left')

# If no match → lawn area = 0
left['lawn_area'] = left['lawn_area'].fillna(0).clip(lower=0)

print(f'  All buildings: {len(left):,}')
print(f'  With lawn area > 0: {(left["lawn_area"] > 0).sum():,}')
print(f'  With lawn area = 0 (no parcel match): {(left["lawn_area"] == 0).sum():,}')
lawn = left['lawn_area']
print(f'\nLawn area stats (all buildings including 0s):')
print(f'  mean   : {lawn.mean():.1f} m2')
print(f'  median : {lawn.median():.1f} m2')
print(f'  max    : {lawn.max():.1f} m2')

Step 5b — Left join for lawn area (all buildings)...
  All buildings: 4,133,323
  With lawn area > 0: 3,698,754
  With lawn area = 0 (no parcel match): 434,569

Lawn area stats (all buildings including 0s):
  mean   : 655.0 m2
  median : 360.6 m2
  max    : 4867696.1 m2


## Step 6 — Geothermal Boolean Flag

Use the **left join** result for the geothermal flag:
- Buildings with lawn area > 0 and lawn area >= required area → True
- Buildings with lawn area = 0 (no parcel) → False (insufficient data)
- Buildings with lawn area < required area → False

In [8]:
lookup_path = os.path.join(OUTPUT_DIR, 'geothermal_lookup.csv')
peak_path   = os.path.join(OUTPUT_DIR, 'peak_heat_power.parquet')

if not os.path.exists(lookup_path):
    print('geothermal_lookup.csv not found — run geothermal_lookup_table.ipynb first')
elif not os.path.exists(peak_path):
    print('peak_heat_power.parquet not found — run annual_to_peak_heat_power.ipynb first')
else:
    lookup  = pd.read_csv(lookup_path)
    peak_df = pd.read_parquet(peak_path, columns=['id', 'peak_heat_power_kw'])

    left = left.merge(peak_df, on='id', how='left')

    # Vectorised lookup using searchsorted
    lookup_sorted = lookup.sort_values('heat_power_kw').reset_index(drop=True)
    power_vals = lookup_sorted['heat_power_kw'].values
    area_vals  = lookup_sorted['area_m2'].values

    peak_vals = left['peak_heat_power_kw'].fillna(0).values
    indices   = np.searchsorted(power_vals, peak_vals)
    indices   = np.clip(indices, 0, len(area_vals)-1)
    left['required_area_m2'] = area_vals[indices].astype(float)
    left.loc[left['peak_heat_power_kw'].isna(), 'required_area_m2'] = np.nan

    # Boolean flag
    left['geothermal_feasible'] = (
        (left['lawn_area'] > 0) &
        (left['lawn_area'] >= left['required_area_m2'])
    )

    n_feasible = left['geothermal_feasible'].sum()
    print(f'Geothermal feasibility (all {len(left):,} buildings):')
    print(f'  Feasible (True) : {n_feasible:,}  ({n_feasible/len(left)*100:.1f}%)')
    print(f'  Not feasible    : {len(left)-n_feasible:,}')
    print()
    print('By size class:')
    for sc in ['SFH', 'TH', 'MFH', 'AB']:
        sub = left[left['size_class'] == sc]
        if sub.empty: continue
        n_f = sub['geothermal_feasible'].sum()
        print(f'  {sc}: {n_f:>8,} feasible out of {len(sub):,}  ({n_f/len(sub)*100:.1f}%)')

Geothermal feasibility (all 4,134,675 buildings):
  Feasible (True) : 3,613,885  (87.4%)
  Not feasible    : 520,790

By size class:
  SFH: 2,461,069 feasible out of 2,723,856  (90.4%)
  TH:  471,733 feasible out of 507,759  (92.9%)
  MFH:  677,025 feasible out of 897,713  (75.4%)
  AB:    4,058 feasible out of 5,347  (75.9%)


## Step 7 — Save Results

In [9]:
# Save inner join result (confirmed residential only)
inner_cols = ['id', 'size_class', 'area_m2', 'lawn_area']
inner_available = [c for c in inner_cols if c in inner.columns]
inner[inner_available].to_parquet(
    os.path.join(OUTPUT_DIR, 'residential_buildings.parquet'), index=False
)
print(f'Saved residential_buildings.parquet: {len(inner):,} buildings')

# Save full result with geothermal flag
out_cols = ['id', 'size_class', 'area_m2', 'lawn_area',
            'geothermal_feasible', 'peak_heat_power_kw', 'required_area_m2']
out_available = [c for c in out_cols if c in left.columns]
left[out_available].to_parquet(
    os.path.join(OUTPUT_DIR, 'geothermal_feasibility.parquet'), index=False
)
print(f'Saved geothermal_feasibility.parquet: {len(left):,} buildings')
print('\nDone.')

Saved residential_buildings.parquet: 3,705,834 buildings
Saved geothermal_feasibility.parquet: 4,134,675 buildings

Done.
